# Special Cases Corrective Pass

A **second-pass, low-throughput** corrective script -- deliberately separate from `GeneDataSourcing.ipynb`, which stays the simple, high-throughput main sourcing path and is not touched here. This notebook re-fetches and re-processes just the handful of genes whose real biology the standard genomic-DNA-to-protein translation gets wrong, and overwrites their (currently broken/empty-isoform) JSON files in `RefGenes/NHGeneBodySupp/` with corrected ones.

Three cases handled today, all confirmed live against real NCBI data this session (see `src/genewriter/special_translation.py` for the full verification notes):

1. **Selenocysteine recoding** -- the ~25-member human selenoproteome (GPX1-4/6, TXNRD1-3, DIO1-3, SELENOP/K/S/M/N/O/F/H/I/T/V/W, MSRB1, SEPHS2). Each has an in-frame `UGA` that gets recoded to selenocysteine (`U`) instead of terminating translation. Standard translation reads `UGA` as stop and truncates the isoform to nothing usable -- confirmed for all 25 genes; SELENOP (10 in-frame `UGA`/Sec codons in one CDS) matched perfectly once fixed.
2. **Non-AUG start codons** -- a small, deliberately incomplete, only-grows-by-direct-confirmation list (`MYC` today) where the real, NCBI-annotated CDS starts with a non-`ATG` codon (e.g. `CTG`), yet the real protein still starts with Met (the initiator tRNA always loads Met, regardless of the literal start codon).
3. **Programmed ribosomal frameshifting** -- `OAZ1` (+1 shift at its ORF1/ORF2 UGA junction) and `PEG10` (-1 shift at the canonical `GGGAAAC` slippery heptamer). Naive translation matches the real protein up to a specific codon, then a 1-nucleotide skip (+1) or overlap (-1) reproduces the rest exactly. Both confirmed full-length, byte-for-byte against live NCBI data.

**Not handled here** (investigated, deliberately not built -- see `special_translation.py`'s module docstring): A-to-I RNA editing (GRIA2/GRIA3/GRIK1/GRIK2/CYFIP2/BLCAP all checked live) and C-to-U editing (APOB). For every one of these, standard genomic translation already matches NCBI's canonical protein exactly -- there's no sourcing bug, just a corpus-completeness gap (the edited variant has no RefSeq record of its own). Stop-codon readthrough not yet investigated with real data.

Uses the exact same (already-fixed, this session) `getSeqFrom`/`locate_codons`/`saveNaturalGeneObj` as `GeneDataSourcing.ipynb` -- copied here rather than imported, since notebook cells aren't importable and the main notebook is intentionally left alone. If that notebook's extraction logic changes again, port the same change here by hand.

**Not yet designed**: once these genes are sourced correctly, the GA/scoring side needs to know not to mutate through a Sec/SECIS site, a non-AUG start, or (now) a frameshift junction -- naive codon-optimization could silently destroy any of these mechanisms (e.g. swapping PEG10's `GGGAAAC` slippery heptamer for a synonymous-but-different sequence would kill the frameshift). No such guard exists yet.

In [ ]:
REPO_DIR = '/content/GeneWriter'
!git clone https://github.com/LukeTheGeneWriter/GeneWriter.git {REPO_DIR}

In [ ]:
import os
!curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
!chmod +x datasets
os.environ['PATH'] += ':/content/datasets:'
!pip install --quiet biopython

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive

In [ ]:
import sys
src_dir = os.path.join(REPO_DIR, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from genewriter.special_translation import (
    SELENOPROTEIN_GENE_IDS,
    NON_ATG_START_GENE_IDS,
    FRAMESHIFT_GENE_IDS,
    check_cds_against_protein_special,
    check_cds_against_protein_at_position,
    check_cds_against_protein_frameshift,
)

%run GeneClassesCloud.ipynb

import subprocess
import re
import json
import shutil
import dataclasses
from time import sleep
import pandas as pd
from Bio import SeqIO

## Core extraction functions

Copied verbatim from `GeneDataSourcing.ipynb`'s current (fixed) `getSeqFrom`/`locate_codons`/`determineProtWeight`/`saveNaturalGeneObj`/`clearDownloadDirectory` -- same off-by-one fix, same batching-collision fix, same `_MAX_SANE_EXON_LEN` guard. Not re-derived, not re-verified here -- if these need to change, change them in `GeneDataSourcing.ipynb` first (where they're proven against real data) and port the change here by hand.

In [ ]:
def determineProtWeight(aaSeq: str):
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
    aaSeq = aaSeq.replace("X", "A")
    prot = ProteinAnalysis(aaSeq)
    return prot.molecular_weight()

def clearDownloadDirectory(dest: str):
  if dest in os.getcwd():
    for f in os.listdir():
      if os.path.isfile(f):
        os.remove(f)
      else:
        shutil.rmtree(f)
  else:
    print("What are tryna delete?")

def saveNaturalGeneObj(obj: NaturalGene):
    print("CWD: ", str(os.getcwd()))
    while "MyDrive" in os.getcwd():
      %cd ..
    %cd MyDrive
    %cd RefGenes
    %cd NHGeneBodySupp

    name = str(obj.geneID) + r'.json'
    fname = name
    b = json.dumps(dataclasses.asdict(obj))
    with open(fname, 'wb') as infile:
        infile.write(b.encode('utf-8'))
    infile.close()
    print("# Closed -- wrote", fname)
    %cd ..
    %cd ..
    %cd WorkingFolders
    %cd NCBIDownload

_MAX_SANE_EXON_LEN = 100_000

def getSeqFrom(genomicAccessionVersion, genomicRange, sequenceName, genomes, exonBeginEndOrder):
  print("GetSeqFrom started for ", sequenceName)
  exonsFull = []
  for genome in genomes:
    if genomicAccessionVersion == genome.id.split(":")[0]:
      range_prelim = genome.id.split(":")[1]
      range_prelim = range_prelim.replace("c", "")
      range_prelim = range_prelim.split("-")
      r1 = int(range_prelim[0])
      r2 = int(range_prelim[1])

      candidate_exons = []
      for ex in exonBeginEndOrder:
        rr1 = int(ex['begin'])
        rr2 = int(ex['end'])
        ord = ex['order']

        claimed_len = abs(rr2 - rr1) + 1
        if claimed_len > _MAX_SANE_EXON_LEN:
          print(f"Skipping exon order {ord} for {sequenceName}: claimed length {claimed_len} exceeds _MAX_SANE_EXON_LEN ({_MAX_SANE_EXON_LEN}).")
          continue

        if rr1 > rr2:
          rnaDirection = 'minus'
        else:
          rnaDirection = 'plus'

        q = None
        if genomicRange['orientation'] == 'plus' and rnaDirection == 'plus':
          scaledBegin = int(rr1) - r1
          scaledEnd = int(rr2) - r1 + 1
          q = genome.seq[scaledBegin:scaledEnd]
        elif genomicRange['orientation'] == 'plus' and rnaDirection == 'minus':
          print("(Case B: not implemented, skipping)")
          continue
        elif genomicRange['orientation'] == 'minus' and rnaDirection == 'plus':
          scaledBegin = r1 - int(rr2)
          scaledEnd = r1 - int(rr1) + 1
          q = genome.seq[scaledBegin:scaledEnd]
        elif genomicRange['orientation'] == 'minus' and rnaDirection == 'minus':
          print("(Case D: not implemented, skipping)")
          continue

        if q is not None and abs(len(q) - claimed_len) > 1:
          print(f"Skipping exon order {ord} for {sequenceName}: computed slice length {len(q)} doesn't match claimed exon length {claimed_len} (off by {abs(len(q) - claimed_len)}).")
          continue

        if q is not None:
          candidate_exons.append(q)

      if candidate_exons:
        exonsFull = candidate_exons
        break

  return exonsFull

def locate_codons(codvec, codingSeq, atg_start, dna_seq, exsupp):
  aa_len = len(codvec)
  clocs = []
  eseq = ''
  for e in exsupp:
    s = e['seq']
    eseq += s
  assert eseq == codingSeq

  r = atg_start
  cnum = 0
  carry = ''
  for e in exsupp:
    elen = len(e['seq'])
    if r > len(e['seq']):
      r -= len(e['seq'])
      continue

    had_carry = carry != ''
    if had_carry:
      r = 3 - len(carry)
      cod = carry + e['seq'][:r]
      remseq = e['seq'][r:]
      carry = ''
      clocs.append((cod, 'S'))
      cnum += 1
      if cod == 'TAA' or cod == 'TAG' or cod == 'TGA' or cnum >= aa_len:
        break
      r = 0
    else:
      remseq = e['seq'][r:]
      r = 0
    if elen - len(remseq) < 15:
      while elen - len(remseq) < 15 and len(remseq) >= 3 and cnum < aa_len:
        clocs.append((remseq[:3], 'F'))
        remseq = remseq[3:]
        cnum += 1
    if len(remseq) > 15:
      while len(remseq) > 15 and cnum < aa_len:
        clocs.append((remseq[:3], 'I'))
        remseq = remseq[3:]
        cnum += 1
    if len(remseq) <= 15:
      while len(remseq) != 0 and cnum < aa_len:
        if len(remseq) < 3:
          carry += remseq
          remseq = ''
          continue
        else:
          clocs.append((remseq[:3], 'T'))
          if len(remseq) == 3:
            remseq = ''
          else:
            remseq = remseq[3:]
          cnum += 1
    if cnum >= aa_len:
      break
  return clocs[:aa_len]

## Corrective processing

Mirrors `GeneDataSourcing.ipynb`'s `_process_gene()` shape (same isoform-refinement passes, same `saveNaturalGeneObj` call), but tries three checks in order: `check_cds_against_protein_special()` (selenocysteine-aware), then `check_cds_against_protein_at_position()` (anchored at NCBI's own annotated CDS start, handles non-AUG starts), then `check_cds_against_protein_frameshift()` (registered +1/-1 shift genes). Composing these handles a gene needing more than one at once (none of today's genes actually do, TXNRD3 needs only the second).

For a gene recovered via the frameshift check, `locate_codons()` itself is *not* modified -- instead `apply_frameshift_to_exons()` pre-splices the real exon-fragment list the same way `apply_frameshift()` splices a flat CDS string, so the existing, unmodified `locate_codons()` walks straight through and lands on the correct codons and genomic positions. See that function's docstring for the one caveat (a cosmetic splice-proximity mistag right at the frameshift junction, not a codon-identity issue).

In [ ]:
def _process_special_gene(gene_id, gene_row, prod_row, proteins, genomes):
  '''Returns True if at least one isoform was built and saved, False otherwise.'''
  try:
    geneName = gene_row['symbol']
    organism = gene_row['commonName']
    chromosome = gene_row['chromosomes']

    isoforms = prod_row['transcripts']
    if not isinstance(isoforms, list):
      print(f"Gene {gene_id} ({geneName}): no transcripts list, skipping.")
      return False

    isoSet = []
    gene_accessions_seen = []
    for iso in isoforms:
      if 'name' in iso:
        isoformNumber = iso['name'].replace('transcript variant ', '')
      else:
        isoformNumber = -1
      try:
        if "X" in isoformNumber:
          continue
      except TypeError:
        pass
      if len(isoforms) == 1:
        isoformNumber = 1

      if 'protein' not in iso:
        continue
      protFile = iso['protein']['accessionVersion']
      associatedProtein = None
      for p in proteins:
        if protFile in p.id:
          pseq = p.seq
          pweight = determineProtWeight(str(pseq))
          associatedProtein = ProteinObj(str(pseq), pweight, gene_id)
      if associatedProtein is None or associatedProtein.aaSeq == '':
        print(f"  Gene {gene_id} isoform {isoformNumber}: no protein record matched {protFile!r}, skipping this isoform.")
        continue

      if 'genomicLocations' not in iso:
        continue

      for gLocSet in iso['genomicLocations']:
        if 'exons' not in gLocSet:
          continue
        exonBeginEndOrder = gLocSet['exons']
        genomicAccessionVersion = gLocSet['genomicAccessionVersion']
        genomicRange = gLocSet['genomicRange']
        sequenceName = gLocSet['sequenceName']
        if genomicAccessionVersion not in gene_accessions_seen:
          gene_accessions_seen.append(genomicAccessionVersion)

        exons = getSeqFrom(genomicAccessionVersion, genomicRange, sequenceName, genomes, exonBeginEndOrder)
        exSupp = []
        cds_seq_from_exons = ''
        for ex in exons:
          exSupp.append({"seq": str(ex)})
          cds_seq_from_exons += str(ex)

        atg_indexes = [m.start() for m in re.finditer('ATG', cds_seq_from_exons)]
        atg_start, codvec = check_cds_against_protein_special(
            cds_seq_from_exons, atg_indexes, associatedProtein.aaSeq, gene_id=gene_id)

        if atg_start is None and 'cds' in iso:
          cds_begin = int(iso['cds']['range'][0]['begin']) - 1
          if check_cds_against_protein_at_position(
              cds_seq_from_exons, cds_begin, associatedProtein.aaSeq, gene_id=gene_id):
            atg_start = cds_begin
            codvec = [cds_seq_from_exons[cds_begin + 3*i:cds_begin + 3*i + 3]
                      for i in range(len(associatedProtein.aaSeq))]
            print(f"  Gene {gene_id} isoform {isoformNumber}: recovered via NCBI-CDS-anchored (non-AUG-start-aware) check.")

        used_frameshift = False
        if atg_start is None:
          fs_atg_start, fs_codvec = check_cds_against_protein_frameshift(
              cds_seq_from_exons, atg_indexes, associatedProtein.aaSeq, gene_id=gene_id)
          if fs_atg_start is not None:
            atg_start, codvec = fs_atg_start, fs_codvec
            used_frameshift = True
            print(f"  Gene {gene_id} isoform {isoformNumber}: recovered via frameshift-aware check.")

        if atg_start is not None:
          if used_frameshift:
            spec = FRAMESHIFT_GENE_IDS[int(gene_id)]
            boundary_pos = atg_start + spec['shift_residue'] * 3
            codon_exsupp = apply_frameshift_to_exons(exSupp, boundary_pos, spec['shift_offset'])
            codon_coding_seq = ''.join(e['seq'] for e in codon_exsupp)
          else:
            codon_exsupp = exSupp
            codon_coding_seq = cds_seq_from_exons
          codlocvec = locate_codons(codvec, codon_coding_seq, atg_start, None, codon_exsupp)
          igb = IsoformGeneBody(isoformNumber, associatedProtein, None, cds_seq_from_exons, codlocvec, -1, exSupp)
          isoSet.append(igb)
        else:
          print(f"  Gene {gene_id} isoform {isoformNumber}: still no valid start found even with special-case handling -- coding sequence {cds_seq_from_exons[:60]}...")

    refinedIsos = []
    for iso in isoSet:
      if iso.geneBody != [] and iso not in refinedIsos:
        refinedIsos.append(iso)
    isoSet = refinedIsos
    refinedIsos = []
    protSeqs = []
    for iso in isoSet:
      if iso.geneBody != [] and str(iso.associatedProtein.aaSeq) not in protSeqs:
        protSeqs.append(str(iso.associatedProtein.aaSeq))
        refinedIsos.append(iso)
    isoSet = refinedIsos
    refinedIsos = []
    exonSeqs = []
    for iso in isoSet:
      if iso.geneBody != []:
        linked = ''.join(exon['seq'] for exon in iso.geneBody)
        if linked not in exonSeqs:
          exonSeqs.append(linked)
          refinedIsos.append(iso)
    isoSet = refinedIsos

    if not isoSet:
      print(f"Gene {gene_id} ({geneName}): still no valid isoform after special-case handling.")
      return False

    gSeq = None
    for acc in gene_accessions_seen:
      matched_genome = next((g for g in genomes if g.id.split(':')[0] == acc), None)
      if matched_genome is not None:
        gSeq = str(matched_genome.seq)
        break
    if gSeq is None:
      gSeq = str(genomes[0].seq) if genomes else ''
    for iso in isoSet:
      iso.fullSequence = gSeq

    ng = NaturalGene(isoSet, gene_id, geneName, organism, gSeq, [], [], {}, chromosome)
    saveNaturalGeneObj(ng)
    print(f"Gene {gene_id} ({geneName}): corrected, {len(isoSet)} isoform(s) saved.")
    return True
  except Exception as e:
    print(f"Gene {gene_id}: unexpected failure during corrective pass: {e!r}")
    import traceback
    traceback.print_exc()
    return False

## Download + sweep

One batched download for the whole sweep list (small, ~26 genes -- no RAM-tripping/batch-splitting logic needed the way the main high-throughput script has).

In [ ]:
def run_corrective_sweep(gene_ids, data_directory):
  '''Downloads and reprocesses every gene in gene_ids with the special-case-
  aware checks. Returns (succeeded, failed) gene ID lists.'''
  while "RefGenes" in os.getcwd():
    %cd ..
  %cd $data_directory
  if "NCBIDownload" not in os.getcwd():
    %cd NCBIDownload

  label = f"special_{gene_ids[0]}-{gene_ids[-1]}"
  fnamezip = f'batch_{label}.zip'
  fname = f'batch_{label}'
  download_result = subprocess.run(
      ['../../../../datasets', 'download', 'gene', 'gene-id', *[str(g) for g in gene_ids],
       '--filename', fnamezip, '--include', 'gene,protein,product-report', '--no-progressbar'],
      capture_output=True, text=True,
  )
  if download_result.returncode != 0:
    print("Error: download failed.")
    print(download_result.stdout)
    print(download_result.stderr)
    return [], list(gene_ids)
  sleep(5)

  unzip_result = subprocess.run(['unzip', '-o', fnamezip, '-d', fname], capture_output=True, text=True)
  if unzip_result.returncode != 0:
    print("Error: unzip failed.")
    print(unzip_result.stdout)
    print(unzip_result.stderr)
    return [], list(gene_ids)

  data_path = os.path.join(fname, 'ncbi_dataset', 'data', 'data_report.jsonl')
  pro_path = os.path.join(fname, 'ncbi_dataset', 'data', 'protein.faa')
  gene_path = os.path.join(fname, 'ncbi_dataset', 'data', 'gene.fna')
  prodrep_path = os.path.join(fname, 'ncbi_dataset', 'data', 'product_report.jsonl')

  proteins = list(SeqIO.parse(pro_path, 'fasta'))
  genomes = list(SeqIO.parse(gene_path, 'fasta'))
  with open(data_path) as f:
    df_final = pd.json_normalize([json.loads(l) for l in f.read().splitlines()])
  with open(prodrep_path) as f:
    df_final1 = pd.json_normalize([json.loads(l) for l in f.read().splitlines()])

  succeeded, failed = [], []
  for gene_id in gene_ids:
    gene_rows = df_final[df_final['geneId'].astype(str) == str(gene_id)]
    prod_rows = df_final1[df_final1['geneId'].astype(str) == str(gene_id)]
    if len(gene_rows) == 0 or len(prod_rows) == 0:
      print(f"Gene {gene_id}: no data_report/product_report row found in this download.")
      failed.append(gene_id)
      continue
    ok = _process_special_gene(gene_id, gene_rows.iloc[0], prod_rows.iloc[0], proteins, genomes)
    (succeeded if ok else failed).append(gene_id)

  # Guarded, not unconditional -- saveNaturalGeneObj() (called inside
  # _process_special_gene, once per successful gene) already leaves cwd back
  # inside NCBIDownload when it's done, so blindly re-running the same %cd
  # dance here would try to descend into a `WorkingFolders` that doesn't
  # exist *inside* NCBIDownload and fail. Only re-navigate if we're not
  # already there (e.g. every gene in this sweep failed before ever calling
  # saveNaturalGeneObj, so cwd never moved from where the unzip left it --
  # still inside NCBIDownload either way, but the guard keeps this correct
  # regardless of which path got us here).
  if "NCBIDownload" not in os.getcwd():
    while "RefGenes" in os.getcwd():
      %cd ..
    %cd $data_directory
    %cd NCBIDownload
  clearDownloadDirectory(r'NCBIDownload')

  return succeeded, failed

## Run

`SWEEP_GENE_IDS` is the literal registry -- add a gene here (and to `special_translation.py`'s `SELENOPROTEIN_GENE_IDS`/`NON_ATG_START_GENE_IDS`, or a new category if a different biological phenomenon comes up) to have this pass cover it.

In [ ]:
SWEEP_GENE_IDS = sorted(set(SELENOPROTEIN_GENE_IDS) | set(NON_ATG_START_GENE_IDS) | set(FRAMESHIFT_GENE_IDS))
print(f"{len(SWEEP_GENE_IDS)} genes in this sweep:", SWEEP_GENE_IDS)

succeeded, failed = run_corrective_sweep(SWEEP_GENE_IDS, r'WorkingFolders')
print(f"\n{len(succeeded)} corrected: {succeeded}")
print(f"{len(failed)} still failing: {failed}")